# AgentMind — PyTorch LoRA Training on Colab

Fine-tune Qwen2.5-0.5B with LoRA on domain-specific data.
GPU runtime required (T4 or better).

Estimated time for tool_caller (2000 steps): **~8-12 minutes**

In [ ]:
# @title 1. Install dependencies
!pip install -q torch transformers peft bitsandbytes datasets accelerate sentencepiece

In [ ]:
# @title 2. Clone repo
import os
REPO_URL = "https://github.com/RajeshShrirao/Agentmind.git"
WORK_DIR = "/content/Agentmind"

if not os.path.exists(WORK_DIR):
    !git clone {REPO_URL} {WORK_DIR}
%cd {WORK_DIR}
print(f"Working directory: {os.getcwd()}")

## 3. Configure Training

Edit the cell below to change training parameters.

In [ ]:
# @title 3. Configure training parameters
import json

CONFIG = {
    "data_path": "data/apprentice_tool_caller.jsonl",
    "domain": "tool_caller",
    "backbone": "Qwen/Qwen2.5-0.5B",
    "steps": 2000,
    "lr": 2e-4,
    "seq_len": 256,
    "seq_len_schedule": {"0": 128, "200": 256},
    "lora_rank": 16,
    "lora_alpha": 32.0,
    "grad_accum": 4,
    "grad_clip": 1.0,
    "output_dir": "/content/drive/MyDrive/AgentMind/checkpoints" if os.path.exists("/content/drive") else "./checkpoints",
}

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# @title 4. Mount Google Drive (optional)
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

## 5. Load Model & Apply LoRA

Loads Qwen2.5-0.5B in 4-bit (NF4), adds special tokens, applies LoRA on 7 target modules.

In [ ]:
# @title 5. Load model and apply LoRA
import torch
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

SPECIAL_TOKENS = [
    "<|tool_call|>", "<|plan|>", "<|memory|>", "<|scratch|>", "<|observe|>",
    "<|think_start|>", "<|think_end|>", "<|system|>", "<|user|>", "<|assistant|>",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "No GPU")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(CONFIG["backbone"], trust_remote_code=True)
tokenizer.add_tokens(SPECIAL_TOKENS)
tokenizer.pad_token = tokenizer.eos_token
print(f"Vocabulary size: {len(tokenizer)}")

# Load model in 4-bit
quant_config = transformers.BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    CONFIG["backbone"],
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)

model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id

# Apply LoRA via PEFT
lora_config = LoraConfig(
    r=CONFIG["lora_rank"],
    lora_alpha=CONFIG["lora_alpha"],
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} ({trainable/1e6:.2f}M) / {total:,} ({total/1e6:.2f}M)")
print(f"Model memory: ~{total * 2 / 1e9:.2f}GB in fp16")

## 6. Generate Data (if missing)

The JSONL dataset is gitignored (too large for GitHub). This cell generates it if absent.

In [ ]:
# @title 6. Generate data if missing
import os
DATA_PATH = CONFIG["data_path"]
if not os.path.exists(DATA_PATH):
    print(f"{DATA_PATH} not found — generating synthetic data...")
    !python generate_scaled_synthetic.py
    print(f"Generated data files:")
    !ls -lh data/apprentice_*.jsonl
else:
    size = os.path.getsize(DATA_PATH)
    print(f"{DATA_PATH} exists ({size/1024/1024:.0f} MB)")

## 7. Prepare Dataset

Loads JSONL, tokenizes with chat template, masks non-assistant tokens with -100.

In [ ]:
# @title 7. Prepare dataset
import json
import random
from torch.utils.data import Dataset, DataLoader

def make_labels(ids, tokenizer):
    labels = [-100] * len(ids)
    in_assistant = False
    im_start_id = tokenizer.convert_tokens_to_ids("<|im_start|>")
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    assistant_id = tokenizer.convert_tokens_to_ids("assistant")
    for i, tok_id in enumerate(ids):
        if tok_id == im_start_id and i + 1 < len(ids) and ids[i + 1] == assistant_id:
            in_assistant = True
        if in_assistant:
            labels[i] = tok_id
        if tok_id == im_end_id:
            in_assistant = False
    return labels


class AgentDataset(Dataset):
    def __init__(self, samples, tokenizer, max_len=1024):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_len = max_len
        self._tokenized = None

    @classmethod
    def from_raw(cls, path, tokenizer, max_len=1024):
        samples = []
        with open(path) as f:
            for line in f:
                samples.append(json.loads(line.strip()))
        ds = cls(samples, tokenizer, max_len)
        ds._tokenize_all()
        return ds

    def _tokenize_all(self):
        tokenized = []
        for sample in self.samples:
            messages = sample["messages"]
            text = self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            ids = self.tokenizer.encode(text)
            labels = make_labels(ids, self.tokenizer)
            tokenized.append((ids, labels))
        self._tokenized = tokenized

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        ids, labels = self._tokenized[idx]
        ids = ids[:self.max_len]
        labels = labels[:self.max_len]
        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }


def collate_fn(batch, pad_token_id=0):
    max_len = max(len(item["input_ids"]) for item in batch)
    input_ids, labels, attention_mask = [], [], []
    for item in batch:
        pad = max_len - len(item["input_ids"])
        input_ids.append(item["input_ids"] + [pad_token_id] * pad)
        labels.append(item["labels"] + [-100] * pad)
        attention_mask.append(item["attention_mask"] + [0] * pad)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }


# Load dataset
dataset = AgentDataset.from_raw(CONFIG["data_path"], tokenizer, max_len=1024)
print(f"Loaded {len(dataset)} samples from {CONFIG['data_path']}")

# Quick validation
sample = dataset[0]
print(f"Sample: input_ids={len(sample['input_ids'])}, labels_mask={sum(1 for l in sample['labels'] if l != -100)}/{len(sample['labels'])} assistant tokens")

## 8. Run Training

LoRA-only training with gradient accumulation, mixed precision, and cosine LR schedule.

In [ ]:
# @title 7. Train
import time
import math

seq_len_schedule = {int(k): v for k, v in CONFIG["seq_len_schedule"].items()}
total_steps = CONFIG["steps"]
grad_accum = CONFIG["grad_accum"]
batch_size = 2  # fits in 15GB GPU with 4-bit

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=0.01)
warmup = min(200, max(1, total_steps // 10))
scheduler = transformers.get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup, num_training_steps=total_steps
)
scaler = torch.cuda.amp.GradScaler()

n_total = len(dataset)
n_val = max(1, n_total // 20)
val_idx = set(random.sample(range(n_total), n_val))
train_idx = [i for i in range(n_total) if i not in val_idx]

opt_step = 0
micro_step = 0
accum_loss = 0.0
t_start = time.time()
last_logged = 0
nan_count = 0
current_seq_len = 0

model.train()

print(f"Starting training: {total_steps} opt-steps, grad_accum={grad_accum}, batch={batch_size}")
print(f"  Effective batch size: {batch_size * grad_accum}")
print()

while opt_step < total_steps:
    # Get seq_len from schedule
    target_seq_len = current_seq_len
    for threshold, length in sorted(seq_len_schedule.items()):
        if opt_step >= threshold:
            target_seq_len = length
    if target_seq_len != current_seq_len:
        print(f"  [seq_len] {current_seq_len} -> {target_seq_len}")
        current_seq_len = target_seq_len

    n_epoch = min(5000, len(train_idx))
    epoch_indices = random.sample(train_idx, n_epoch)
    dataset.max_len = current_seq_len

    subset = torch.utils.data.Subset(dataset, epoch_indices)
    loader = DataLoader(
        subset, batch_size=batch_size, shuffle=True,
        collate_fn=lambda b: collate_fn(b, pad_token_id=tokenizer.pad_token_id or 0),
    )

    for batch in loader:
        if opt_step >= total_steps:
            break

        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attn_mask = batch["attention_mask"].to(device)

        with torch.cuda.amp.autocast():
            outputs = model(
                input_ids=input_ids,
                labels=labels,
                attention_mask=attn_mask,
            )
            loss = outputs.loss / grad_accum

        if not torch.isfinite(loss):
            nan_count += 1
            optimizer.zero_grad()
            continue

        accum_loss += outputs.loss.item()
        scaler.scale(loss).backward()
        micro_step += 1

        if micro_step % grad_accum == 0:
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["grad_clip"])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()

            avg_loss = accum_loss / grad_accum
            accum_loss = 0.0

            if opt_step % 100 == 0 or opt_step == total_steps - 1:
                elapsed = time.time() - t_start
                steps_since = opt_step - last_logged + 1
                tok_s = (input_ids.shape[1] * grad_accum * steps_since) / (elapsed + 1e-8)
                lr_now = scheduler.get_last_lr()[0]
                print(f"[{CONFIG['domain']}] step {opt_step:3d}/{total_steps} "
                      f"loss {avg_loss:.4f} lr {lr_now:.2e} "
                      f"grad_norm {grad_norm:.3f} {tok_s:.0f} tok/s")
                t_start = time.time()
                last_logged = opt_step

            opt_step += 1

print(f"\nTraining complete: {opt_step} steps, {nan_count} NaN batches")

## 9. Save Adapter

Saves the LoRA adapter weights and tokenizer.

In [ ]:
# @title 8. Save adapter
import os
output_dir = CONFIG["output_dir"]
adapter_dir = f"{output_dir}/adapters/{CONFIG['domain']}"
os.makedirs(adapter_dir, exist_ok=True)

model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved adapter to {adapter_dir}")

# Show file sizes
for f in os.listdir(adapter_dir):
    path = os.path.join(adapter_dir, f)
    if os.path.isfile(path):
        size = os.path.getsize(path)
        print(f"  {f}: {size/1024:.1f} KB")

## 10. Quick Test

Test the trained adapter with a sample prompt.

In [ ]:
# @title 9. Test inference
from transformers import pipeline

prompt = "What tools do you have access to?"
messages = [
    {"role": "system", "content": "You are a helpful assistant with tool access."},
    {"role": "user", "content": prompt},
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.7,
        do_sample=True,
    )

response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=False)
print(f"Prompt: {prompt}")
print(f"Response: {response}")